# Выводы по проделанному анализу

## 1. Введение и цели исследования
Целью данного проекта являлась разработка математически обоснованного, устойчивого и автоматизированного алгоритма распределения верхнеуровневых корпоративных планов компании по конверсии и среднему количеству товаров в чеке (UPT) для розничной сети, состоящей из 502 магазинов. 

Традиционный подход к планированию ("уравниловка" или пропорциональное увеличение от факта прошлого года) часто приводит к демотивации персонала в исторически слабых локациях и недополучению прибыли в сильных. Настоящее исследование решает задачу построения индивидуальных плановых профилей (Bottom-Up подход) с их последующей жесткой увязкой с макро-целями бизнеса (Top-Down калибровка).


## 2. Методологический стек и этапы исследования
### Этап 2.1. Первичный анализ и предобработка данных (EDA)
В ходе разведочного анализа исходных данных (111 258 строк) была проведена фильтрация технических аномалий и выбросов:
* Идентифицированы и обработаны строки с нулевым трафиком и некорректными математическими соотношениями (покупатели > посетителей, продажи < покупателей).
* Апрель 2025 года содержит данные только за 1–17 числа, в связи с чем он был корректно исключен из анализа сезонности как неполный месяц, но зафиксирован как маркер оперативного краткосрочного тренда.
* Найдена сильная дифференциация метрик в разрезе Категорий магазинов (наивысшая конверсия традиционно у формата `Outlet` — до `16.5%`, наименьшая — у категории `A` — `11.6%`) и Федеральных Округов (ФО), что подтвердило невозможность использования единого плоского плана для всей сети.

### Этап 2.2. Анализ стабильности магазинов и профилирование
Для изоляции локальной силы каждого магазина от общесетевых колебаний были введены метрики `Conversion_Index` и `UPT_Index`:
$$Index = \frac{\text{Метрика конкретного магазина за месяц}}{\text{Средняя метрика компании за этот же месяц}}$$

Анализ устойчивости данных индексов через расчет матрицы корреляции Пирсона между аналогичными периодами 2024 и 2025 годов показал следующие результаты:
* **UPT Индекс** продемонстрировал высочайшую линейную связь (r = `0.9033`). Это доказывает, что глубина чека — внутренний операционный показатель, слабо зависящий от внешних макроэкономических факторов, локации или трафика. Он полностью определяется внутренними стандартами обслуживания, работой персонала и мерчандайзингом, оставаясь стабильным из года в год.
* **Конверсия Индекс** показал умеренно-высокую связь (r = `0.7771`). Данная метрика более волатильна, так как напрямую зависит от маркетинговой активности конкурентов, погоды и качества входящего трафика.

**Вывод этапа**: Высокая автокорреляция индексов подтвердила базовую гипотезу проекта — индивидуальный исторический профиль магазина является надежным фундаментом для прогнозирования будущих периодов.


## 3. Архитектура финальной модели и алгоритм калибровки

Реализованная в Python модель связывает воедино очищенный массив данных и корпоративные цели без использования хардкода:

1. **Динамический импорт таргетов:** Из датафрейма `df_goals` цели компании автоматически переводятся в словарь. При изменении бизнес-целей в исходном файле модель пересчитается автоматически.
2. **Развертывание матрицы:** Генерируется плоская таблица, содержащая уникальные комбинации `ID магазина` × `Прогнозный месяц (Май–Август)`.
3. **Расчет первоначального плана (Initial Plan):**
$$InitialPlanConv = \text{TargetConv} \times \text{ConvIndex}$$
$$InitialPlanUPT = \text{TargetUPT} \times \text{UPTIndex}$$

1. **Контроль сходимости (Check и Калибровка):**
Математически, простое перемножение индексов на уровне Bottom-Up не сходится с планом корпоративного центра из-за структурных сдвигов и разного масштаба магазинов. Для устранения этого дисбаланса модель рассчитывает помесячный коэффициент калибровки:
$$CalibFactor = \frac{CorporateTarget}{InitialPlanMean}$$

1. **Финальный расчет (Final Plan):**
$$FinalPlan = InitialPlan \times CalibFactor$$


## 4. Визуализация и интерпретация результатов

Для валидации модели были построены графики распределения финальных планов (Boxplot) в сопоставлении с линиями корпоративных таргетов.

### Ключевые выводы по графикам:
1. **Индивидуализация планов:** Графики наглядно иллюстрируют уход от "плоского" планирования. Модель сформировала "облака" (коробки) планов вокруг целевой линии компании. Исторически сильные магазины получили более амбициозные задачи, а слабые — выполнимые плановые показатели, соответствующие их локации.
2. **Математическая точность калибровки:** Медианы получившихся распределений (коробок) для каждого месяца идеально совпали с красными пунктирными линиями корпоративных целей. Это визуально доказывает, что этап калибровки (Check) отработал без искажений: индивидуальность розничных точек сохранена, но цели всей компании по итогу будут гарантированно достигнуты.